# Day 021 — Exercise 5: ai_tag_file

**What you'll build:** `ai_tag_file(content, model)` — uses a local LLM to categorize a file's text content, returning `{category, tags, summary}`.

**Why it matters:** This is the AI layer in the batch pipeline. Once you can tag one file, you can tag thousands by looping with `batch_process_files`.

In [ ]:
import ollama
import json

## Your Implementation

In [ ]:
def ai_tag_file(content: str, model: str = "llama3.2") -> dict:
    """
    Use a local LLM to categorize file content.

    Args:
        content: The text content to categorize.
        model:   Ollama model name.

    Returns:
        dict with keys:
            category — one word: technical/personal/financial/creative/other
            tags     — list of up to 5 keyword strings
            summary  — one sentence describing the content
    """
    # TODO: build a system prompt asking for JSON with category/tags/summary
    # TODO: call ollama.chat with format='json'
    # TODO: json.loads the response content
    # TODO: return {'category': ..., 'tags': ..., 'summary': ...}
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'ai_tag_file' in globals()
        passed += 1; print('\u2705 Check 1: ai_tag_file defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    result = None
    SAMPLE = (
        'This Python script processes a list of numbers and computes '
        'their average, standard deviation, and median values.'
    )

    # Check 2: returns a dict (1 LLM call)
    try:
        result = ai_tag_file(SAMPLE)
        assert isinstance(result, dict), \
            f'expected dict, got {type(result)}'
        passed += 1; print('\u2705 Check 2: returns a dict')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: has required keys
    try:
        assert result is not None, 'result is None (Check 2 failed)'
        for key in ('category', 'tags', 'summary'):
            assert key in result, f"missing key '{key}': {result}"
        passed += 1; print('\u2705 Check 3: dict has category/tags/summary keys')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: category is a non-empty string
    try:
        assert result is not None, 'result is None'
        cat = result['category']
        assert isinstance(cat, str) and len(cat) > 0, \
            f'category must be non-empty str, got {cat!r}'
        passed += 1; print(f'\u2705 Check 4: category is a string ({cat!r})')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: tags is a list
    try:
        assert result is not None, 'result is None'
        tags = result['tags']
        assert isinstance(tags, list), \
            f'tags must be list, got {type(tags)}'
        passed += 1; print(f'\u2705 Check 5: tags is a list ({tags})')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def ai_tag_file(content: str, model: str = "llama3.2") -> dict:
    system = (
        "You are a file categorization assistant. "
        "Given text content, return JSON with exactly these keys: "
        "category (one word: technical, personal, financial, creative, or other), "
        "tags (list of up to 5 keyword strings), "
        "summary (one sentence describing the content). "
        "Return only valid JSON."
    )
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": f"Categorize this content:\n\n{content[:2000]}"},
        ],
        format="json",
    )
    raw = response["message"]["content"]
    try:
        data = json.loads(raw)
    except Exception:
        data = {}
    return {
        "category": str(data.get("category", "other")).lower().strip() or "other",
        "tags": list(data.get("tags", [])),
        "summary": str(data.get("summary", "")),
    }
```

</details>